# Analysis of Benchmark Results for 100 Single-Shot v9_combined_23_disrupt HDF5 Files using HSDS

This notebook benchmarks reading the Alcator C-Mod shot data for the v9_combined_23_disrupt MIT signal selection from two storage formats -- HDF5 files imported into HSDS -- on two AWS EC2 instances using AWS S3 storage. This top cell is a self-contained TL;DR that summarises the entire notebook; the analysis sections that follow it provide the per-format details, the side-by-side comparison plots, and all benchmark numbers.

## Data: HDF5 Files

Each HDF5 file was created with the following properties:

* Each file holds approximately 370 signals from a single shot. The signal selection was done by the MIT team and designated as v9_combined_23_disrupt.
* The files have same HDF5 group hierarchy: /shots/<SHOT_ID>/signals/<signal MDSplus expression>.
* The MDSplus dim_of data are stored as HDF5 dimension scale datasets in the /shots/<SHOT_ID>/ group and attached to the appropriate dimensions of the signal HDF5 datasets.
* Only unique dimension scales are stored by comparing the MD5 checksum of dimension scale values. This means there could be multiple signal datasets that share the same dimension scale.
* HDF5 datasets (dimension scales and signals) with total size greater than 8 kB are compressed with the Deflate (a.k.a. gzip) compression at level 4.

## About HSDS

HSDS (HDF Scalable Data Service) is a REST-based service desgined for the reading and writing of HDF5 data.  In HSDS traditional HDF5
files are stored as sharded collections of JSON objects (for metadata) and binary blobs (for chunk data).  This storage layout enables efficient reads and writes to AWS S3 stores (though POSIX storage is also supported).  HSDS runs as a set of front-end (the SN nodes)
and back-end (the DN nodes) processes.  The SN nodes provide horizontal scaling (more SN nodes can support more clients) while the DN
nodes are used to parallize individual client requests.  The number of SN and DN nodes can be configured at start up time subject
to the available memory and CPU resources of the machine.

HSDS clients can use the REST API directly, the h5pyd package (for Python), or the REST VOL (for C/C++).  This study has focused on Python clients using h5pyd.

## Comparison of old vs new HSDS release when Accessing Single-Shot HDF5 files in AWS S3

HSDS was initially designed to support files that contained relatively few (but potentially quite large) datasets.  By contrast,
the data collected from CMOD, contains a large number of smaller datasets (i.e. signals).  This resulted in poor performance due to
each read or write of a signal requiring an seperate HTTP requests.

The forthcoming HSDS release 1.0 (as of writing this) along with the corresponding h5pyd client library improves performance
by batching multiple operations in a single request, thus reducing latency.  Write performance is improved by enabling multiple updates (e.g. creating mulitple datasets) to be done in a single http requests.  Read performance is improved by the use of <I>consolidated metadata objects</I>.  These are JSON objects that contain all the metadata for a given file.  HSDS will create these objects in a background task when the file has not been modified recently.  On file open, the h5pyd client will retrieve the consolidated metadata (if available), cache it and use it to return results for any future api read operations that don't require dataset data.

This notebook compares benchmark results between the latest current HSDS release and a development version of future 1.0 release. The goal is to assess whether there are significant performance differences between the old and new code. 

## Benchmarks

The benchmarks were run on two AWS EC2 instances:

* m5.4xlarge with 16 vCPU (8 CPU cores, 2 threads per core), 64 GiB memory, up to 10 Gbps network bandwidth.
* m7i.48xlarge with 192 vCPUs, 768 GiB memory, 50 Gbps network bandwidth, 40 Gbps EBS bandwidth.

For the m5.4xlarge instance, HSDS was configured to run as 4 SN and 4 DN containers in Docker.

For the m7i.48xlarge instance, HSDS was configured to run as 16 SN and 16 DN containers in Docker.

In both cases, HSDS the data was stored on a S3 bucket (in the same region as the EC2 instances).

Data reading tasks were shared among a range of Dask workers, never to exceed one worker per vCPU. The processing tasks were divided using an equal effort principle, and the actual number of Dask workers was adjusted accordingly so not to have any unused workers.  The HSDS endpoint used by each worker was: http://localhost:n, where n is the worker number modulo the number of open ports (4 for the smaller instance, 16 for the larger).  The resulted in a fairly even distribution of the load among the available SN containers.

The benchmark task read all signals from all the files / stores in one of two access patterns:
* One signal at a time from all the files / stores (obj-type="signals"). The number of files to read a signal from varied per Dask worker depending on their total number.
* Read all signals from one file / store at a time (obj-type="shots"). The number of files to read all the signals from varied per Dask worker depending on their total number.

## TL;DR Conclusions

* Importing data using hsload was approximately 2x faster with the new HSDS version
* With one worker, performance improved by 30%
* With more workers, the improvement was greater with a 82% improvement using 16 workers
* Beyond 16 workers, HSDS (both the old and the new versions) with 4 SN and 4 DN nodes was not able to handle the client load and the test failed

The overall conclusion is the new HSDS release is a signficant improvement for working with files that contain many smaller objects.  The new design reduces latency by sending more information per request, so that the total number of requests needed is less.  The improvement is even greater as the number of workers increases (up to a point).  With more than 16 workers, the HSDS configuration would need to be scaled up (say to 8 SN and 8 DN nodes), but the system we ran the test on was limited to 8 CPU cores and 64 GB of memory.

---

## Benchmark Data Analysis

In [1]:
import pandas as pd
import hvplot.pandas  # noqa: F401
import holoviews as hv

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

hv.extension("bokeh")

In [2]:
# the "bifull" csv are from the large EC2 instance,
# the other csv file are from the smaller EC2 instance
new_csv_file = "./bench.hsdsv1.bigfull.csv"  # "./bench.hsdsv1.csv"
old_csv_file = "./bench.hsdsv09.bigfull.csv"  # "./bench.hsdsv09.csv"

Read benchmark data for the new HSDS service:

In [3]:
data_4x_new = pd.read_csv(new_csv_file)
data_4x_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 39716 entries, 0 to 39715
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   worker#              39716 non-null  int64  
 1   obj-id               39716 non-null  str    
 2   open+read-data-time  39716 non-null  float64
 3   wrkr-num-objs        39716 non-null  int64  
 4   mean-obj-time        39716 non-null  float64
 5   num-dsets            39716 non-null  int64  
 6   mean-dset-time       39716 non-null  float64
 7   num-workers          39716 non-null  int64  
 8   obj-type             39716 non-null  str    
 9   tot-num-obj          39716 non-null  int64  
 10  total-runtime        39716 non-null  float64
dtypes: float64(4), int64(5), str(2)
memory usage: 3.9 MB


Read benchmark data for the old HSDS:

In [4]:
data_4x_old = pd.read_csv(old_csv_file)
data_4x_old.info()

<class 'pandas.DataFrame'>
RangeIndex: 39709 entries, 0 to 39708
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   worker#              39709 non-null  int64  
 1   obj-id               39709 non-null  str    
 2   open+read-data-time  39709 non-null  float64
 3   wrkr-num-objs        39709 non-null  int64  
 4   mean-obj-time        39709 non-null  float64
 5   num-dsets            39709 non-null  int64  
 6   mean-dset-time       39709 non-null  float64
 7   num-workers          39709 non-null  int64  
 8   obj-type             39709 non-null  str    
 9   tot-num-obj          39709 non-null  int64  
 10  total-runtime        39709 non-null  float64
dtypes: float64(4), int64(5), str(2)
memory usage: 3.9 MB


In [5]:
data_4x_old.groupby(["num-workers"]).describe()

worker#                                                       \
               count       mean        std  min    25%   50%    75%   max   
num-workers                                                                 
1                1.0   1.000000        NaN  1.0   1.00   1.0   1.00   1.0   
2                2.0   1.500000   0.707107  1.0   1.25   1.5   1.75   2.0   
4                4.0   2.500000   1.290994  1.0   1.75   2.5   3.25   4.0   
8             2001.0   4.488756   2.287793  1.0   2.00   4.0   6.00   8.0   
16            3822.0   8.131868   4.415645  1.0   4.00   8.0  12.00  16.0   
24            5316.0  11.172498   6.238062  1.0   6.00  11.0  16.00  24.0   
32            6747.0  14.094709   7.990085  1.0   7.00  14.0  21.00  32.0   
48            9326.0  19.428051  11.277258  1.0  10.00  19.0  29.00  48.0   
64           12490.0  25.688551  14.777991  1.0  13.00  25.0  38.00  63.0   

            open+read-data-time                                             \
                          count       mean       std        min        25%   
num-workers                                                                  
1                           1.0  69.382739       NaN  69.382739  69.382739   
2                           2.0  35.386170  9.431220  28.717291  32.051731   
4                           4.0  17.627783  4.181138  11.386838  17.225211   
8                        2001.0   5.226443  1.655754   0.378213   4.521100   
16                       3822.0   3.021152  1.073734   0.238823   2.572493   
24                       5316.0   3.582836  2.449768   0.234704   2.218496   
32                       6747.0   3.835237  2.246719   0.253456   2.407284   
48                       9326.0   3.894857  8.955203   0.236334   1.913468   
64                      12490.0   3.868683  7.537481   0.287710   1.753926   

                                              wrkr-num-objs             \
                   50%        75%         max         count       mean   
num-workers                                                              
1            69.382739  69.382739   69.382739           1.0   1.000000   
2            35.386170  38.720610   42.055050           2.0   1.000000   
4            19.473246  19.875818   20.177801           4.0   1.000000   
8             4.981440   5.693154   19.788882        2001.0  12.104448   
16            2.961775   3.428090   13.673361        3822.0   6.472004   
24            2.862213   4.044551   33.582896        5316.0   4.706922   
32            3.282833   4.613444   34.916388        6747.0   3.749963   
48            2.616069   4.254877  378.342847        9326.0   2.771928   
64            2.315955   4.011871  283.318472       12490.0   1.993034   

                                                   mean-obj-time             \
                  std  min   25%   50%   75%   max         count       mean   
num-workers                                                                   
1                 NaN  1.0   1.0   1.0   1.0   1.0           1.0  69.382739   
2            0.000000  1.0   1.0   1.0   1.0   1.0           2.0  35.386170   
4            0.000000  1.0   1.0   1.0   1.0   1.0           4.0  17.627783   
8            2.088441  1.0  12.0  13.0  13.0  14.0        2001.0   0.450052   
16           1.339757  1.0   7.0   7.0   7.0   8.0        3822.0   0.476191   
24           0.761168  1.0   5.0   5.0   5.0   6.0        5316.0   0.767790   
32           0.577674  1.0   4.0   4.0   4.0   5.0        6747.0   1.019464   
48           0.485709  1.0   3.0   3.0   3.0   3.0        9326.0   1.392387   
64           0.236814  1.0   2.0   2.0   2.0   3.0       12490.0   1.945980   

                                                                               \
                  std        min        25%        50%        75%         max   
num-workers                                                                     
1                 NaN  69.382739  69.382739  69.382739  69.382739   69

In [6]:
data_4x_new.groupby(["num-workers"]).describe()


worker#                                                       \
               count       mean        std  min    25%   50%    75%   max   
num-workers                                                                 
1                1.0   1.000000        NaN  1.0   1.00   1.0   1.00   1.0   
2                2.0   1.500000   0.707107  1.0   1.25   1.5   1.75   2.0   
4                4.0   2.500000   1.290994  1.0   1.75   2.5   3.25   4.0   
8             2001.0   4.488756   2.287793  1.0   2.00   4.0   6.00   8.0   
16            3822.0   8.131868   4.415645  1.0   4.00   8.0  12.00  16.0   
24            5316.0  11.172498   6.238062  1.0   6.00  11.0  16.00  24.0   
32            6747.0  14.094709   7.990085  1.0   7.00  14.0  21.00  32.0   
48            9326.0  19.428051  11.277258  1.0  10.00  19.0  29.00  48.0   
64           12497.0  25.708330  14.797534  1.0  13.00  26.0  38.00  64.0   

            open+read-data-time                                             \
                          count       mean       std        min        25%   
num-workers                                                                  
1                           1.0  59.500254       NaN  59.500254  59.500254   
2                           2.0  28.644450  1.941719  27.271448  27.957949   
4                           4.0  14.295650  6.762291   5.713806  11.241174   
8                        2001.0   4.142581  2.552443   0.625095   2.094656   
16                       3822.0   1.622016  0.809940   0.150924   1.119020   
24                       5316.0   1.333714  0.636435   0.133473   0.916260   
32                       6747.0   1.164648  0.669993   0.145995   0.798535   
48                       9326.0   1.043865  0.565538   0.138650   0.699911   
64                      12497.0   0.921238  0.548297   0.147614   0.601543   

                                             wrkr-num-objs             \
                   50%        75%        max         count       mean   
num-workers                                                             
1            59.500254  59.500254  59.500254           1.0   1.000000   
2            28.644450  29.330952  30.017453           2.0   1.000000   
4            14.810099  17.864575  21.848596           4.0   1.000000   
8             3.296294   5.632993  18.363229        2001.0  12.104448   
16            1.388691   2.021640   9.080734        3822.0   6.472004   
24            1.261890   1.609311   8.106940        5316.0   4.706922   
32            1.073408   1.394821  10.826734        6747.0   3.749963   
48            0.965408   1.253437   7.595101        9326.0   2.771928   
64            0.805099   1.067527   6.692043       12497.0   1.992478   

                                                   mean-obj-time             \
                  std  min   25%   50%   75%   max         count       mean   
num-workers                                                                   
1                 NaN  1.0   1.0   1.0   1.0   1.0           1.0  59.500254   
2            0.000000  1.0   1.0   1.0   1.0   1.0           2.0  28.644450   
4            0.000000  1.0   1.0   1.0   1.0   1.0           4.0  14.295650   
8            2.088441  1.0  12.0  13.0  13.0  14.0        2001.0   0.356119   
16           1.339757  1.0   7.0   7.0   7.0   8.0        3822.0   0.255819   
24           0.761168  1.0   5.0   5.0   5.0   6.0        5316.0   0.285180   
32           0.577674  1.0   4.0   4.0   4.0   5.0        6747.0   0.310790   
48           0.485709  1.0   3.0   3.0   3.0   3.0        9326.0   0.375779   
64           0.237911  1.0   2.0   2.0   2.0   3.0       12497.0   0.463906   

                                                                              \
                  std        min        25%        50%        75%        max   
num-workers                                                                    
1                 NaN  59.500254  59.500254  59.500254  59.500254  59.500254   
2    

These were the benchmark parameter combinations:

In [7]:
data_4x_old[["obj-type", "num-workers"]].drop_duplicates()

,obj-type,num-workers
0,shots,1
1,shots,2
3,shots,4
7,signals,8
2000,shots,8
2008,signals,16
5814,shots,16
5830,signals,24
11122,shots,24
11146,signals,32


In [8]:
data_4x_new[["obj-type",  "num-workers"]].drop_duplicates()

,obj-type,num-workers
0,shots,1
1,shots,2
3,shots,4
7,signals,8
2000,shots,8
2008,signals,16
5814,shots,16
5830,signals,24
11122,shots,24
11146,signals,32


Column `obj-type` describes data access type during one benchmark run:

* `obj-type = shots` means all signals from one shot file were read.
* `obj-type = signals` means all signals from all the files were read, one at a time.


Since the two ways of reading data by `shots` and `signals` are so different they cannot be compared to each other. Separate them into different DataFrames.

In [9]:
old_4x_shots = data_4x_old[data_4x_old["obj-type"] == "shots"]
new_4x_shots = data_4x_new[data_4x_new["obj-type"] == "shots"]
old_4x_signals = data_4x_old[data_4x_old["obj-type"] == "signals"]
new_4x_signals = data_4x_new[data_4x_new["obj-type"] == "signals"]

In [10]:
new_4x_shots.head()

,worker#,obj-id,open+read-data-time,wrkr-num-objs,mean-obj-time,num-dsets,mean-dset-time,num-workers,obj-type,tot-num-obj,total-runtime
0,1,1160929006,59.500254,1,59.500254,786,0.075700,1,shots,100,59.519708
1,1,1160929011,30.017453,1,30.017453,416,0.072157,2,shots,100,30.038312
2,2,1160929011,27.271448,1,27.271448,379,0.071956,2,shots,100,30.038312
3,1,1160928013,5.713806,1,5.713806,203,0.028147,4,shots,100,21.870577
4,2,1160928013,16.536567,1,16.536567,212,0.078003,4,shots,100,21.870577


In [11]:
new_4x_signals.head()

,worker#,obj-id,open+read-data-time,wrkr-num-objs,mean-obj-time,num-dsets,mean-dset-time,num-workers,obj-type,tot-num-obj,total-runtime
7,1,p_to_v_expr,6.155987,13,0.473537,13,0.473537,8,signals,250,1034.143299
8,2,p_to_v_expr,6.154376,13,0.473414,13,0.473414,8,signals,250,1034.143299
9,3,p_to_v_expr,6.325976,13,0.486614,13,0.486614,8,signals,250,1034.143299
10,4,p_to_v_expr,6.154752,13,0.473442,13,0.473442,8,signals,250,1034.143299
11,5,p_to_v_expr,5.390288,13,0.414638,13,0.414638,8,signals,250,1034.143299


### Total Runtime and Peformance

Total benchmark runtime in the `tot-runtime` column is the elapsed time of the entire benchmark as measured by the main process. The total runtime encompasses:
1. Dividing data access plan across Dask workers and their intialization.
1. All Dask workers completing their jobs.
1. Collecting Dask worker benchmark data.

Below are four DataFrames with total runtimes separated for the signal and shot benchmarks. Their runtimes are so different that there is no point comparing them together. The new DataFrames include several original columns plus a new column `norm-tot-runtime`. This column holds computed performance ratios to the _baseline_ benchmark. The baseline benchmark is one of the available benchmarks selected because it represents the most common set of libhdf5 settings, compute resources, and data access. The baseline benchmarks are:

* HSDS Domains:
    * Reading all signals for a shot: 1 Dask worker
    * Reading all signals for all the shots: 8 Dask worker

In [12]:
old_shots_4x_runtime = old_4x_shots[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
old_shots_4x_runtime["where"] = "S3"
old_shots_4x_runtime["norm-tot-runtime"] = (
    old_shots_4x_runtime.loc[0, "total-runtime"] / old_shots_4x_runtime["total-runtime"]
)

old_signals_4x_runtime = old_4x_signals[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
old_signals_4x_runtime["where"] = "S3"
old_signals_4x_runtime["norm-tot-runtime"] = (
    old_signals_4x_runtime.loc[0, "total-runtime"] / old_signals_4x_runtime["total-runtime"]
)

new_shots_4x_runtime = new_4x_shots[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
new_shots_4x_runtime["where"] = "S3"
new_shots_4x_runtime["norm-tot-runtime"] = (
    new_shots_4x_runtime.loc[0, "total-runtime"] / new_shots_4x_runtime["total-runtime"]
)

new_signals_4x_runtime = new_4x_signals[
    ["num-workers", "total-runtime"]
].drop_duplicates(ignore_index=True)
new_signals_4x_runtime["where"] = "S3"
new_signals_4x_runtime["norm-tot-runtime"] = (
    new_signals_4x_runtime.loc[0, "total-runtime"] / new_signals_4x_runtime["total-runtime"]
)

### Reading All Data from a Single Shot File

Plots of performance ratio and runtime when reading all data from a single shot file:

In [13]:
plot_kwargs = {
    "x": "num-workers",
}
(
    old_shots_4x_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * old_shots_4x_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
    * hv.HLine(1).opts(line_width=0.7, color="pink")
    * new_shots_4x_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * new_shots_4x_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
).options(
    legend_position="top_right",
    title="Shot File: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Performance ratio (>1 better)",
    xlim=(0, old_shots_4x_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
) + (
    old_shots_4x_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * old_shots_4x_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
    * new_shots_4x_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * new_shots_4x_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
).options(
    legend_position="bottom_right",
    title="Shot File: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Total runtime / [s]",
    xlim=(0, old_shots_4x_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    show_grid=True,
    height=400,
    width=500,
)

:Layout
   .Overlay.I  :Overlay
      .Curve.I    :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.I  :Scatter   [num-workers]   (norm-tot-runtime)
      .HLine.I    :HLine   [x,y]
      .Curve.II   :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.II :Scatter   [num-workers]   (norm-tot-runtime)
   .Overlay.II :Overlay
      .Curve.I    :Curve   [num-workers]   (total-runtime)
      .Scatter.I  :Scatter   [num-workers]   (total-runtime)
      .Curve.II   :Curve   [num-workers]   (total-runtime)
      .Scatter.II :Scatter   [num-workers]   (total-runtime)

### Reading All Signals from All Shot Files

Plots of performance ratio and runtime when reading all signals from all the shot files:

In [14]:
plot_kwargs = {
    "x": "num-workers",
}
(
    old_signals_4x_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * old_signals_4x_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
    * hv.HLine(1).opts(line_width=0.7, color="pink")
    * new_signals_4x_runtime.hvplot.line(y="norm-tot-runtime", **plot_kwargs)
    * new_signals_4x_runtime.hvplot.scatter(y="norm-tot-runtime", **plot_kwargs)
).options(
    legend_position="top_right",
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Performance ratio (>1 better)",
    xlim=(0, old_signals_4x_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
) + (
    old_signals_4x_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * old_signals_4x_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
    * new_signals_4x_runtime.hvplot.line(y="total-runtime", **plot_kwargs)
    * new_signals_4x_runtime.hvplot.scatter(y="total-runtime", **plot_kwargs)
).options(
    legend_position="bottom_right",
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    xlabel="Number of Dask workers",
    ylabel="Total runtime / [s]",
    xlim=(0, old_signals_4x_runtime["num-workers"].max() + 1),
    ylim=(0, None),
    show_grid=True,
    height=400,
    width=500,
)

:Layout
   .Overlay.I  :Overlay
      .Curve.I    :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.I  :Scatter   [num-workers]   (norm-tot-runtime)
      .HLine.I    :HLine   [x,y]
      .Curve.II   :Curve   [num-workers]   (norm-tot-runtime)
      .Scatter.II :Scatter   [num-workers]   (norm-tot-runtime)
   .Overlay.II :Overlay
      .Curve.I    :Curve   [num-workers]   (total-runtime)
      .Scatter.I  :Scatter   [num-workers]   (total-runtime)
      .Curve.II   :Curve   [num-workers]   (total-runtime)
      .Scatter.II :Scatter   [num-workers]   (total-runtime)

### Display Some Worker Mean Runtimes

The `mean-obj-time` column holds mean read times of _objects_ in a shot file. Which _object_ is read depends on the `obj-type` column, with the values `shots` or `signals`.

In [15]:
(
    old_4x_signals.hvplot.box(
        y="mean-obj-time",
        by=["num-workers"],
    )
    * new_4x_signals.hvplot.box(
        y="mean-obj-time",
        by=["num-workers"],
    )
).options(
    # ylim=(8, 11),
    title="All Signals All Shots: Old (blue) vs New (red) HSDS",
    height=400,
    show_legend=False,
    xlabel="Number of Dask workers",
    ylabel="Worker mean signal read time / [seconds]",
    show_grid=True,
)

:Overlay
   .BoxWhisker.I  :BoxWhisker   [num-workers]   (mean-obj-time)
   .BoxWhisker.II :BoxWhisker   [num-workers]   (mean-obj-time)

In [16]:
new_4x_signals.groupby(["num-workers"])["mean-obj-time"].describe()

,count,mean,std,min,25%,50%,75%,max
num-workers,,,,,,,,
8,1993.0,0.342776,0.201471,0.124740,0.164514,0.364868,0.446338,1.412556
16,3806.0,0.249237,0.109934,0.133771,0.170246,0.223838,0.296308,1.374618
24,5292.0,0.281185,0.122049,0.133473,0.198239,0.262864,0.330170,1.621388
32,6715.0,0.307218,0.162548,0.142635,0.224019,0.280011,0.355731,2.706683
48,9282.0,0.372939,0.181656,0.138650,0.267044,0.342742,0.431887,2.531700
64,12433.0,0.462335,0.272165,0.142846,0.303330,0.402590,0.533055,3.346022


In [17]:
old_4x_signals.groupby(["num-workers"])["mean-obj-time"].describe()

,count,mean,std,min,25%,50%,75%,max
num-workers,,,,,,,,
8,1993.0,0.430730,0.114997,0.289818,0.362263,0.393178,0.467686,1.522222
16,3806.0,0.465101,0.134300,0.238823,0.390189,0.438460,0.507243,1.953337
24,5292.0,0.758997,0.499264,0.234704,0.470310,0.600393,0.858914,6.716579
32,6715.0,1.014132,0.562755,0.253456,0.651986,0.857164,1.194349,8.729097
48,9282.0,1.384989,3.000621,0.236334,0.697402,0.925818,1.495267,126.114282
64,12433.0,1.942599,3.783790,0.287710,0.883793,1.154915,2.003861,141.659236


In [18]:
old_4x_signals.groupby(["num-workers"]).describe()

worker#                                                     \
               count       mean        std  min   25%   50%   75%   max   
num-workers                                                               
8             1993.0   4.488710   2.287779  1.0   2.0   4.0   6.0   8.0   
16            3806.0   8.130321   4.414745  1.0   4.0   8.0  12.0  16.0   
24            5292.0  11.166478   6.234144  1.0   6.0  11.0  16.0  24.0   
32            6715.0  14.083246   7.981962  1.0   7.0  14.0  21.0  32.0   
48            9282.0  19.413488  11.268099  1.0  10.0  19.0  28.0  48.0   
64           12433.0  25.673369  14.768169  1.0  13.0  25.0  38.0  63.0   

            open+read-data-time                                          \
                          count      mean       std       min       25%   
num-workers                                                               
8                        1993.0  5.226294  1.651028  0.378213  4.521218   
16                       3806.0  3.020761  1.071755  0.238823  2.573077   
24                       5292.0  3.586809  2.453608  0.234704  2.219271   
32                       6715.0  3.843324  2.248465  0.253456  2.411737   
48                       9282.0  3.899321  8.975982  0.236334  1.911674   
64                      12433.0  3.874117  7.554112  0.287710  1.752695   

                                            wrkr-num-objs             \
                  50%       75%         max         count       mean   
num-workers                                                            
8            4.982751  5.680194   19.788882        1993.0  12.149022   
16           2.961775  3.426095   13.673361        3806.0   6.495008   
24           2.861639  4.052482   33.582896        5292.0   4.723734   
32           3.290596  4.619298   34.916388        6715.0   3.763068   
48           2.614570  4.260041  378.342847        9282.0   2.780328   
64           2.313176  4.016982  283.318472       12433.0   1.997587   

                                                   mean-obj-time            \
                  std  min   25%   50%   75%   max         count      mean   
num-workers                                                                  
8            1.970253  1.0  12.0  13.0  13.0  14.0        1993.0  0.430730   
16           1.294627  1.0   7.0   7.0   7.0   8.0        3806.0  0.465101   
24           0.720688  1.0   5.0   5.0   5.0   6.0        5292.0  0.758997   
32           0.546884  1.0   4.0   4.0   4.0   5.0        6715.0  1.014132   
48           0.471249  1.0   3.0   3.0   3.0   3.0        9282.0  1.384989   
64           0.227587  1.0   2.0   2.0   2.0   3.0       12433.0  1.942599   

                                                                           \
                  std       min       25%       50%       75%         max   
num-workers                                                                 
8            0.114997  0.289818  0.362263  0.393178  0.467686    1.522222   
16           0.134300  0.238823  0.390189  0.438460  0.507243    1.953337   
24           0.499264  0.234704  0.470310  0.600393  0.858914    6.716579   
32           0.562755  0.253456  0.651986  0.857164  1.194349    8.729097   
48           3.000621  0.236334  0.697402  0.925818  1.495267  126.114282   
64           3.783790  0.287710  0.883793  1.154915  2.003861  141.659236   

            num-dsets                                                    \
                count       mean       std  min   25%   50%   75%   max   
num-workers                                                               
8              1993.0  24.076769  6.338876  2.0  24.0  26.0  26.0  42.0   
16             3806.0  12.869417  3.690671  1.0  12.0  14.0  14.0  21.0   
24             5292.0   9.360166  2.391455  1.0   8.0  10.0  10.0  15.0   
32             6715.0   7.455398  1.869490  2.0   6.0   8.0   8.0  12.0   
48             9282.0   5.506464  1.454926  1.0   4.0   6.0   6.0   9.0   
64            1

---